In [87]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets, models
from sklearn.metrics import accuracy_score, f1_score, classification_report
from PIL import Image
import torch.nn as nn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchmetrics.classification import BinaryF1Score

In [88]:
class AlbumentationsImageFolder(torch.utils.data.Dataset):
    def __init__(self, root, transform=None):
        self.folder = datasets.ImageFolder(root=str(root), transform=None)
        self.transform = transform

    def __len__(self):
        return len(self.folder)

    def __getitem__(self, idx):
        path, target = self.folder.samples[idx]
        img = Image.open(path).convert("RGB")
        img = np.array(img)  # albumentations 以 numpy array 输入
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, target

In [89]:
def read_image_labels(csv_path: str) -> pd.DataFrame:
    """Read train.csv and index by image name."""
    df = pd.read_csv(csv_path).set_index("image")
    return df


def get_single_labels(unique_labels) -> List[str]:
    """Split multi-label strings and return list of unique classes."""
    single_labels = []
    for label in unique_labels:
        single_labels += label.split()
    single_labels = set(single_labels)
    return list(single_labels)


def get_one_hot_encoded_labels(dataset_df: pd.DataFrame) -> pd.DataFrame:
    """Convert 'labels' column into multi-hot columns."""
    df = dataset_df.copy()
    unique_labels = df.labels.unique()
    column_names = get_single_labels(unique_labels)

    # 初始化列
    df[column_names] = 0

    # one-hot / multi-hot
    for label in unique_labels:
        label_indices = df[df["labels"] == label].index
        splited_labels = label.split()
        df.loc[label_indices, splited_labels] = 1

    return df

In [90]:
BATCH=16
DATA_ROOT = "./data"
TEST_DIR = os.path.join(DATA_ROOT, "manual_test_images")
TEST_DATA_FILE = os.path.join(DATA_ROOT, "test_split.csv")
CLASSES = [
    "rust",
    "complex",
    "healthy",
    "powdery_mildew",
    "scab",
    "frog_eye_leaf_spot",
]

folders = dict(
    {
        "data": DATA_ROOT,
        "test": TEST_DIR,
    }
)

In [91]:
def get_image(image_id, kind: str = "train") -> Image.Image:
    """Load an image from file."""
    fname = os.path.join(folders[kind], image_id)
    return Image.open(fname)

class PlantDataset(Dataset):
    def __init__(
        self,
        image_ids: pd.Series,
        targets: np.ndarray,
        transform=None,
        target_transform=None,
        kind: str = "train",
    ):
        self.image_ids = image_ids.reset_index(drop=True)
        self.targets = targets
        self.transform = transform
        self.target_transform = target_transform
        self.kind = kind

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # load and transform image
        img = np.array(get_image(self.image_ids.iloc[idx], kind=self.kind))

        if self.transform:
            img = self.transform(image=img)["image"]

        # target
        target = self.targets[idx]
        if self.target_transform:
            target = self.target_transform(target)

        return img, target

In [92]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path settings
model_path_best_valid = Path("data/inception_v3_bestmodel/inception_v3_bestmodel_epoch38.pth")

val_transform = A.Compose(
    [
        A.Resize(height=299, width=299),
        A.Normalize(),        # 默认 ImageNet mean/std
        ToTensorV2(),
    ]
)

In [93]:
test_df = read_image_labels(TEST_DATA_FILE)
test_df = get_one_hot_encoded_labels(test_df)
X_test, Y_test = (
    pd.Series(test_df.index),
    np.array(test_df[CLASSES]),
)
test_dataset = PlantDataset(X_test, Y_test, transform=val_transform, kind="test")
testloader = DataLoader(
    test_dataset, batch_size=BATCH, shuffle=False, num_workers=4
)

num_classes = len(CLASSES)
class_names = CLASSES
print(f"Found {len(test_dataset)} test images, {num_classes} classes: {class_names}")



Found 1864 test images, 6 classes: ['rust', 'complex', 'healthy', 'powdery_mildew', 'scab', 'frog_eye_leaf_spot']


In [ ]:
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_classes = len(CLASSES)

# 1) 按训练时结构建模型：aux_logits=False
model = models.inception_v3(
    pretrained=False,
    aux_logits=False,
    transform_input=True,
).to(DEVICE)

# 2) 覆盖成你训练时的 fc（多标签头）
model.fc = nn.Sequential(
    nn.Linear(2048, 2048),
    nn.ReLU(inplace=True),
    nn.Dropout(0.5),
    nn.Linear(2048, num_classes),
    nn.Sigmoid(),
).to(DEVICE)

# 3) 载入 checkpoint
model_path = model_path_best_valid
ckpt = torch.load(str(model_path), map_location="cpu")

if isinstance(ckpt, dict):
    print("Checkpoint is a dict.")
    if "model" in ckpt:
        state = ckpt["model"]
    elif "model_state_dict" in ckpt:
        state = ckpt["model_state_dict"]
    elif "state_dict" in ckpt:
        state = ckpt["state_dict"]
    else:
        state = ckpt
else:
    print("Checkpoint is a state_dict.")
    state = ckpt

# 去掉 module.
state = {k.replace("module.", ""): v for k, v in state.items()}

# 关键：删掉所有 AuxLogits 权重（因为训练没用它）
state = {k: v for k, v in state.items() if not k.startswith("AuxLogits.")}

# 4) 加载（现在 strict=True 应该完全匹配）
model.load_state_dict(state, strict=True)
print("Loaded model weights successfully.")

model.eval()

test_metric = BinaryF1Score(threshold=0.4).to(DEVICE)

/home/erie_lab/miniconda3/envs/kw465/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/erie_lab/miniconda3/envs/kw465/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
/home/erie_lab/miniconda3/envs/kw465/lib/python3.10/site-packages/torchvision/models/inception.py:43: FutureWarning: The default weight initialization of inception_v3 will be changed in future releases of torchvision. If you wish to keep the old behavior (which leads to long initialization times due to scipy/scipy#11299), please set init_weights=True.
  warnings.warn(


Checkpoint is a dict.
Loaded model weights successfully.


In [95]:
THRESH = 0.4

test_f1 = 0.0
test_batches = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        preds = model(images.float())
        test_f1 += test_metric(preds, labels).item()
        test_batches += 1

avg_test_f1 = test_f1 / max(test_batches, 1)
print(f"Test F1 Score: {avg_test_f1:.4f}")

Test F1 Score: 0.9093


In [96]:
from torchmetrics.classification import MultilabelF1Score, MultilabelAccuracy

num_labels = len(CLASSES)

f1_micro = MultilabelF1Score(num_labels=num_labels, threshold=0.4, average="micro").to(DEVICE)
f1_macro = MultilabelF1Score(num_labels=num_labels, threshold=0.4, average="macro").to(DEVICE)
f1_weighted = MultilabelF1Score(num_labels=num_labels, threshold=0.4, average="weighted").to(DEVICE)

acc_micro = MultilabelAccuracy(num_labels=num_labels, threshold=0.4, average="micro").to(DEVICE)

f1_micro.reset(); f1_macro.reset(); f1_weighted.reset(); acc_micro.reset()

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(DEVICE).float()
        labels = labels.to(DEVICE).float()

        preds = model(images)  # 已经 sigmoid 过了，shape [B,C]

        f1_micro.update(preds, labels)
        f1_macro.update(preds, labels)
        f1_weighted.update(preds, labels)
        acc_micro.update(preds, labels)

print("Overall micro-acc:", acc_micro.compute().item())
print("Overall micro-F1:", f1_micro.compute().item())
print("Overall macro-F1:", f1_macro.compute().item())
print("Overall weighted-F1:", f1_weighted.compute().item())

Overall micro-acc: 0.9666487574577332
Overall micro-F1: 0.9071446061134338
Overall macro-F1: 0.8984600901603699
Overall weighted-F1: 0.9067392349243164


In [97]:
f1_per_class = MultilabelF1Score(num_labels=num_labels, threshold=0.4, average=None).to(DEVICE)
acc_per_class = MultilabelAccuracy(num_labels=num_labels, threshold=0.4, average=None).to(DEVICE)

f1_per_class.reset(); acc_per_class.reset()

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(DEVICE).float()
        labels = labels.to(DEVICE).float()
        preds = model(images)

        f1_per_class.update(preds, labels)
        acc_per_class.update(preds, labels)

f1_c = f1_per_class.compute().cpu().numpy()   # [C]
acc_c = acc_per_class.compute().cpu().numpy() # [C]

print("\nPer-class metrics:")
for i, name in enumerate(CLASSES):
    print(f"{name:20s} acc={acc_c[i]:.4f} f1={f1_c[i]:.4f}")


Per-class metrics:
rust                 acc=0.9802 f1=0.9180
complex              acc=0.9399 f1=0.7477
healthy              acc=0.9925 f1=0.9851
powdery_mildew       acc=0.9941 f1=0.9540
scab                 acc=0.9560 f1=0.9257
frog_eye_leaf_spot   acc=0.9372 f1=0.8602
